# Dos hipotesis por tarea + validacion del ahorro del scoring (4B, L4x4)

Diagnostico de TTT_PLAN 47: **generamos UNA hipotesis y la repetimos 16 veces** (las 16 vistas
son simetrias; en 6 de 12 tareas con pool proponen una sola forma de salida). Los ganadores
generan >=10 hipotesis distintas y verifican.

**Pregunta B (la decisiva)**: dos TTT con semillas de augmentacion distintas, 8 vistas cada uno,
producen pools **diferentes**? Y sobre todo, la **union** contiene la GT mas veces que cada uno
por separado? Prerreg: `union - max(A,B) >= 2` de 12 => el injerto de dos hipotesis esta vivo;
jaccard alto y union ~ A => los dos TTT convergen => idea muerta antes de gastar un slot.

**Pregunta A (la financiacion)**: el re-scoring con 4 augmentaciones en vez de 8 conserva el
top-2? Prerreg: >=90% identico => el ahorro de -4% de reloj es seguro, y es lo que paga el +6,2%
de la doble hipotesis para dejarla en +2,2%, dentro del techo de +4,8%.

**Memoria**: el snapshot del adaptador (~1 GB) vive en **RAM de CPU**, no en GPU.


In [ ]:
%%writefile /kaggle/working/_worker_src.py
import hashlib, json, math, os, random, sys, time
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
T00 = time.time()

SHARD = int(os.environ.get("SHARD", "0"))
N_SHARDS = int(os.environ.get("N_SHARDS", "4"))
N_TAREAS = int(os.environ.get("N_TAREAS", "12"))
N_VISTAS = int(os.environ.get("N_VISTAS", "8"))
BUDGET = float(os.environ.get("BUDGET_VISTA", "45"))
SUCIAS = set(json.loads(os.environ["SUCIAS"]))
OUT = "/kaggle/working/shard%d.json" % SHARD

import numpy as np
import torch
from peft import (LoraConfig, get_peft_model, get_peft_model_state_dict,
                  set_peft_model_state_dict)
from torch.nn.attention import SDPBackend, sdpa_kernel
from transformers import AutoModelForCausalLM

# el montaje de la competicion NO es estable: se busca (leccion del 5-sep)
FICH = "arc-agi_evaluation_challenges.json"
DATA = None
for cand in ("/kaggle/input/competitions/arc-prize-2026-arc-agi-2",
             "/kaggle/input/arc-prize-2026-arc-agi-2"):
    if os.path.exists(os.path.join(cand, FICH)):
        DATA = cand
        break
if DATA is None:
    for root, dirs, files in os.walk("/kaggle/input"):
        if FICH in files:
            DATA = root
            break
assert DATA, "no se encontro %s; /kaggle/input: %s" % (FICH, os.listdir("/kaggle/input"))
CHAL = json.load(open(os.path.join(DATA, FICH)))
SOLS = json.load(open(os.path.join(DATA, "arc-agi_evaluation_solutions.json")))
print("[s%d] datos en %s" % (SHARD, DATA), flush=True)

MDIR = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "config.json" in files and any(x.endswith(".safetensors") for x in files):
        MDIR = root
        break
assert MDIR, "no se encontro el modelo"

_SDPA = [SDPBackend.EFFICIENT_ATTENTION, SDPBackend.FLASH_ATTENTION, SDPBackend.MATH]
IM_START, USER, ASSIST, NL, EOS = 14, 11, 12, 10, 15
ARC_TOKENS = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15]
_TOKID = torch.tensor(ARC_TOKENS, dtype=torch.long, device="cuda:0")
THR = -float(np.log(0.2))          # identico a max_score de produccion
MAX_SEQ, LR = 3500, 5e-5


def coste(t):
    return sum(len(g["input"]) * len(g["input"][0]) + len(g["output"]) * len(g["output"][0])
               for g in t["train"])


# >=3 pares de train para que al retirar uno queden al menos 2 demostraciones
KEYS_ALL = sorted([k for k in CHAL if k not in SUCIAS],
                  key=lambda k: coste(CHAL[k]))[:N_TAREAS]
MIS_KEYS = KEYS_ALL[SHARD::N_SHARDS]
print("[s%d] %d tareas de %d" % (SHARD, len(MIS_KEYS), len(KEYS_ALL)), flush=True)


def grid_tokens(g):
    out = []
    for i, row in enumerate(g):
        out.extend(int(c) for c in row)
        if i < len(g) - 1:
            out.append(NL)
    return out


def pair_tokens_mask(inp, out):
    t = [IM_START, USER, NL] + grid_tokens(inp) + [EOS]
    m = [0] * len(t)
    head = [IM_START, ASSIST, NL]
    body = grid_tokens(out) + [EOS]
    return t + head + body, m + [0] * len(head) + [1] * len(body)


def query_tokens(inp):
    return [IM_START, USER, NL] + grid_tokens(inp) + [EOS] + [IM_START, ASSIST, NL]


def d4(g, k):
    a = np.array(g)
    if k >= 4:
        a = a.T
    for _ in range(k % 4):
        a = np.rot90(a)
    return a.tolist()


def d4_inv(g, k):
    a = np.array(g)
    for _ in range(k % 4):
        a = np.rot90(a, -1)
    if k >= 4:
        a = a.T
    return a.tolist()


def aplicar(g, k, perm):
    return [[perm[c] for c in fila] for fila in d4(g, k)]


def invertir(g, k, perm):
    inv = [0] * 10
    for i, p in enumerate(perm):
        inv[p] = i
    return d4_inv([[inv[c] for c in fila] for fila in g], k)


def build_vistas_train(pares, seed, n_perm=16):
    # 8 D4 x 16 permutaciones = 128 secuencias: calendario EXACTO de produccion
    rng = random.Random(seed)
    perms = [list(range(10))] + [rng.sample(range(10), 10) for _ in range(n_perm - 1)]
    vistas = []
    for k in range(8):
        for perm in perms:
            vistas.append({"k": k, "perm": perm,
                           "orden": rng.sample(range(len(pares)), len(pares))})
    return vistas


def build_vistas_dec(pares, seed, n):
    rng = random.Random(seed)
    v = []
    for i in range(n):
        if i == 0:
            v.append({"k": 0, "perm": list(range(10)), "orden": list(range(len(pares)))})
        else:
            v.append({"k": rng.randrange(8), "perm": rng.sample(range(10), 10),
                      "orden": rng.sample(range(len(pares)), len(pares))})
    return v


base = AutoModelForCausalLM.from_pretrained(
    MDIR, torch_dtype=torch.bfloat16, device_map="cuda:0", attn_implementation="sdpa")
LCFG = LoraConfig(r=256, lora_alpha=32, use_rslora=True, lora_dropout=0.0, bias="none",
                  task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj",
                                  "up_proj", "down_proj", "embed_tokens", "lm_head"])
MODEL = get_peft_model(base, LCFG)
for _, p_ in MODEL.named_parameters():
    if p_.requires_grad:
        p_.data = p_.data.float()
MODEL.enable_input_require_grads()
MODEL.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
# MEMORIA: el snapshot del adaptador vive en RAM de CPU, no en GPU. Es ~1 GB de pesos
# duplicados que solo se usan para resetear entre tareas; produccion los deja en GPU
# (arc_solver.py:317). La copia CPU->GPU del reset cuesta ~0,1 s contra 150-900 s de tarea.
INIT = {k: v.detach().clone().cpu() for k, v in get_peft_model_state_dict(MODEL).items()}
_mem_init = torch.cuda.memory_allocated() / 1e9
try:
    import bitsandbytes as bnb
    OPT_CLS = bnb.optim.AdamW8bit
except Exception:
    OPT_CLS = torch.optim.AdamW
print("[s%d] modelo listo mem=%.1fGB con el snapshot del adaptador en RAM (%.0fs)"
      % (SHARD, _mem_init, time.time() - T00), flush=True)


@torch.no_grad()
def prefill(toks, chunk=512):
    cache, out = None, None
    for i in range(0, len(toks), chunk):
        with sdpa_kernel(_SDPA):
            out = MODEL(input_ids=torch.tensor([toks[i:i + chunk]], device="cuda:0",
                                               dtype=torch.long),
                        past_key_values=cache, use_cache=True, return_dict=True)
        cache = out.past_key_values
    return out.logits[:, -1], cache


def toks_to_grid(toks):
    rows, row = [], []
    for t in toks:
        if t == NL:
            rows.append(row)
            row = []
        elif 0 <= t <= 9:
            row.append(t)
        else:
            return None
    if row:
        rows.append(row)
    if not rows or not rows[0] or any(len(r) != len(rows[0]) for r in rows):
        return None
    return rows


@torch.no_grad()
def dfs(prompt, depth, budget):
    t0 = time.time()
    last, cache0 = prefill(prompt)
    leaves, path = [], []

    def rec(logits, score, cache, left):
        if time.time() - t0 > budget or left <= 1:
            return
        lf = logits.float()
        nll = (score + torch.logsumexp(lf, dim=-1, keepdim=True)
               - lf.index_select(-1, _TOKID)).cpu()
        cands = []
        for j, t in enumerate(ARC_TOKENS):
            sc = nll[0, j].item()
            if sc < THR:
                if t == EOS:
                    leaves.append((sc, list(path)))
                else:
                    cands.append((sc, t))
        for sc, t in sorted(cands):
            if time.time() - t0 > budget:
                return
            with sdpa_kernel(_SDPA):
                o = MODEL(input_ids=torch.tensor([[t]], device="cuda:0", dtype=torch.long),
                          past_key_values=cache, use_cache=True, return_dict=True)
            path.append(t)
            rec(o.logits[:, -1], sc, o.past_key_values, left - 1)
            path.pop()

    rec(last, 0.0, cache0, depth)
    return leaves


def decodificar_pool(demos, consulta, vistas):
    """Devuelve {clave_json_de_rejilla: mejor_beam_score} en el marco ORIGINAL."""
    pool = {}
    for v in vistas:
        prompt = []
        for idx in v["orden"]:
            inp, out = demos[idx]
            t, _ = pair_tokens_mask(aplicar(inp, v["k"], v["perm"]),
                                    aplicar(out, v["k"], v["perm"]))
            prompt += t
        prompt += query_tokens(aplicar(consulta, v["k"], v["perm"]))
        try:
            hojas = dfs(prompt, 931, BUDGET)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            hojas = []
        for sc, camino in sorted(hojas, key=lambda x: x[0])[:8]:
            g = toks_to_grid(camino)
            if g is None:
                continue
            try:
                orig = invertir(g, v["k"], v["perm"])
            except Exception:
                continue
            kk = json.dumps(orig)
            if kk not in pool or sc < pool[kk]:
                pool[kk] = sc
        torch.cuda.empty_cache()
    return pool


@torch.no_grad()
def nll_de(demos, consulta, grid, v):
    """NLL de `grid` como respuesta, bajo la vista v. Replica calc_scores de produccion."""
    toks, mask = [], []
    for idx in v["orden"]:
        inp, out = demos[idx]
        t, m = pair_tokens_mask(aplicar(inp, v["k"], v["perm"]),
                                aplicar(out, v["k"], v["perm"]))
        toks += t
        mask += m
    q = query_tokens(aplicar(consulta, v["k"], v["perm"]))
    cuerpo = grid_tokens(aplicar(grid, v["k"], v["perm"])) + [EOS]
    toks += q + cuerpo
    mask += [0] * len(q) + [1] * len(cuerpo)
    if len(toks) > MAX_SEQ + 1200:
        return None
    ids = torch.tensor([toks], device="cuda:0", dtype=torch.long)
    with sdpa_kernel(_SDPA):
        out = MODEL(input_ids=ids, return_dict=True, use_cache=False)
    lg = out.logits[0, :-1].float()
    tgt = ids[0, 1:]
    m = torch.tensor(mask[1:], device="cuda:0", dtype=torch.bool)
    lp = lg.log_softmax(-1).gather(-1, tgt.unsqueeze(-1)).squeeze(-1)
    return float(-(lp[m].sum()).item())


def entrenar(task_pares, vistas):
    MODEL.config.use_cache = False
    MODEL.train()
    opt = OPT_CLS([p for p in MODEL.parameters() if p.requires_grad], lr=LR)
    seqs = []
    for v in vistas:
        toks, mask = [], []
        for idx in v["orden"]:
            inp, out = task_pares[idx]
            t, m = pair_tokens_mask(aplicar(inp, v["k"], v["perm"]),
                                    aplicar(out, v["k"], v["perm"]))
            toks += t
            mask += m
        if len(toks) <= MAX_SEQ:
            seqs.append((toks, mask))
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(len(seqs), 1))
    for toks, mask in seqs:
        ids = torch.tensor([toks], device="cuda:0", dtype=torch.long)
        lab = torch.tensor([[t if m else -100 for t, m in zip(toks, mask)]],
                           device="cuda:0", dtype=torch.long)
        with sdpa_kernel(_SDPA):
            out = MODEL(input_ids=ids, labels=lab)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_([p for p in MODEL.parameters() if p.requires_grad], 1.0)
        opt.step()
        sch.step()
        opt.zero_grad(set_to_none=True)
    del opt, sch
    MODEL.zero_grad(set_to_none=True)
    torch.cuda.empty_cache()
    MODEL.eval()
    MODEL.config.use_cache = True
    return len(seqs)


RES = {"filas": [], "shard": SHARD}
for i, k in enumerate(MIS_KEYS):
    task = CHAL[k]
    demos = [(p["input"], p["output"]) for p in task["train"]]
    consulta = task["test"][0]["input"]
    gt = SOLS[k][0]
    gt_key = json.dumps(gt)
    seed = int(hashlib.md5(k.encode()).hexdigest()[:8], 16) % 1024**2

    pools, tiempos = {}, {}
    for etiq, off in (("A", 0), ("B", 500000)):
        set_peft_model_state_dict(MODEL, dict(INIT))   # INIT esta en CPU
        vtr = build_vistas_train(demos, seed + off)
        assert len(vtr) == 128, "el TTT debe ver 128 secuencias"
        t0 = time.time()
        try:
            entrenar(demos, vtr)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            print("[s%d] OOM en TTT %s de %s" % (SHARD, etiq, k), flush=True)
            pools[etiq] = {}
            tiempos[etiq] = (0, 0)
            continue
        t_ttt = time.time() - t0
        t0 = time.time()
        pools[etiq] = decodificar_pool(demos, consulta,
                                       build_vistas_dec(demos, seed + off + 1, N_VISTAS))
        tiempos[etiq] = (round(t_ttt, 1), round(time.time() - t0, 1))

    A, B = pools.get("A", {}), pools.get("B", {})
    union = set(A) | set(B)
    inter = set(A) & set(B)

    # --- pregunta del SCORING: el top-2 con 4 augmentaciones = el de 8? (sobre el pool A) ---
    sc8 = sc4 = top_igual = None
    if A:
        vs = build_vistas_dec(demos, seed + 7, 8)
        filas_sc = []
        for kk in list(A)[:12]:                      # tope de 12 candidatos por coste
            g = json.loads(kk)
            nlls = [nll_de(demos, consulta, g, v) for v in vs]
            nlls = [x for x in nlls if x is not None]
            if len(nlls) == 8:
                filas_sc.append((kk, sum(nlls), sum(nlls[:4])))
        if len(filas_sc) >= 2:
            t8 = [x[0] for x in sorted(filas_sc, key=lambda z: z[1])[:2]]
            t4 = [x[0] for x in sorted(filas_sc, key=lambda z: z[2])[:2]]
            sc8, sc4 = t8, t4
            top_igual = set(t8) == set(t4)

    fila = {"task": k, "n_A": len(A), "n_B": len(B), "n_union": len(union),
            "n_interseccion": len(inter),
            "gt_A": gt_key in A, "gt_B": gt_key in B, "gt_union": gt_key in union,
            "jaccard": round(len(inter) / max(len(union), 1), 3),
            "scoring_top2_igual": top_igual,
            "gt_en_top2_8": (gt_key in sc8) if sc8 else None,
            "gt_en_top2_4": (gt_key in sc4) if sc4 else None,
            "t_A": tiempos.get("A"), "t_B": tiempos.get("B")}
    RES["filas"].append(fila)
    print("[s%d %d/%d] %s: A=%d B=%d union=%d jac=%.2f | GT A=%s B=%s UNION=%s | top2(4)==top2(8): %s"
          % (SHARD, i + 1, len(MIS_KEYS), k, len(A), len(B), len(union), fila["jaccard"],
             fila["gt_A"], fila["gt_B"], fila["gt_union"], top_igual), flush=True)
    json.dump(RES, open(OUT, "w"))

json.dump(RES, open(OUT, "w"))
print("[s%d] fin %.0fs" % (SHARD, time.time() - T00), flush=True)


In [ ]:
import json, os, subprocess, sys, time
T0 = time.time()
SUCIAS = ["0934a4d8", "136b0064", "16b78196", "981571dc", "aa4ec2a5", "da515329"]
open("/kaggle/working/worker.py", "w").write(open("/kaggle/working/_worker_src.py").read())
procs = []
for i in range(4):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i), SHARD=str(i), N_SHARDS="4",
               N_TAREAS="12", N_VISTAS="8", BUDGET_VISTA="45",
               SUCIAS=json.dumps(SUCIAS), PYTHONUNBUFFERED="1")
    p = subprocess.Popen([sys.executable, "/kaggle/working/worker.py"], env=env,
                         stdout=open("/kaggle/working/log%d.txt" % i, "w"),
                         stderr=subprocess.STDOUT)
    procs.append(p)
    print("lanzado worker %d (GPU %d)" % (i, i), flush=True)
for i, p in enumerate(procs):
    p.wait()
    print("worker %d rc=%s (%.0fs)" % (i, p.returncode, time.time() - T0), flush=True)

filas = []
for i in range(4):
    f = "/kaggle/working/shard%d.json" % i
    if os.path.exists(f):
        try:
            filas += json.load(open(f))["filas"]
        except Exception as e:
            print("shard %d ilegible: %s" % (i, e), flush=True)
json.dump({"filas": filas}, open("/kaggle/working/ttt_2hip.json", "w"))

for i in range(4):
    print("")
    print("=" * 16 + (" LOG WORKER %d " % i) + "=" * 16, flush=True)
    try:
        print(open("/kaggle/working/log%d.txt" % i).read()[-2500:], flush=True)
    except Exception as e:
        print("sin log: %s" % e, flush=True)

n = len(filas)
gA = sum(1 for f in filas if f["gt_A"])
gB = sum(1 for f in filas if f["gt_B"])
gU = sum(1 for f in filas if f["gt_union"])
jac = [f["jaccard"] for f in filas if f["n_union"]]
ig = [f["scoring_top2_igual"] for f in filas if f["scoring_top2_igual"] is not None]
print("")
print("===== PREGUNTA B: dos TTT dan hipotesis DISTINTAS? =====", flush=True)
print("%-10s %6s %6s %8s %8s %7s %7s %9s" % ("tarea","|A|","|B|","|union|","jaccard","GT A","GT B","GT union"), flush=True)
for f in sorted(filas, key=lambda x: x["task"]):
    print("%-10s %6d %6d %8d %8.2f %7s %7s %9s" %
          (f["task"], f["n_A"], f["n_B"], f["n_union"], f["jaccard"],
           f["gt_A"], f["gt_B"], f["gt_union"]), flush=True)
print("")
print("GT en A: %d/%d | GT en B: %d/%d | GT en la UNION: %d/%d" % (gA,n,gB,n,gU,n), flush=True)
print("jaccard medio: %.2f  (1,00 = hipotesis IDENTICAS, 0 = disjuntas)" % (sum(jac)/max(len(jac),1)), flush=True)
print("PRERREG: union - max(A,B) >= 2 de 12 => dos hipotesis SUMAN cobertura => injerto vivo", flush=True)
print("         jaccard alto y union ~ A => los dos TTT CONVERGEN => idea muerta", flush=True)
print("")
print("===== PREGUNTA A: el scoring 8->4 conserva el top-2? =====", flush=True)
print("top-2 identico con 4 augs que con 8: %d/%d" % (sum(1 for x in ig if x), len(ig)), flush=True)
print("PRERREG: >=90%% identico => el ahorro de -4%% de reloj es seguro y financia lo demas", flush=True)
print("RESULT_JSON:" + json.dumps({"n": n, "gt_A": gA, "gt_B": gB, "gt_union": gU,
      "scoring_igual": sum(1 for x in ig if x), "scoring_n": len(ig),
      "wall_s": round(time.time() - T0)}), flush=True)
